<a href="https://colab.research.google.com/github/somboro08/Hlang/blob/main/mini_gpt_bariba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-GPT Bariba ↔ Français — à partir de zéro (Colab)

Ce notebook prend le tokenizer et les datasets tokenisés déjà préparés
(`tokenizer.json`, `lm_*.bin`, `*_tokenized_*.jsonl`) et construit un petit
GPT en PyTorch : couche d'embedding, embedding de position, blocs
transformer, tête de sortie.

**Avant de lancer :** uploade dans le même dossier (ou dans Google Drive
puis adapte les chemins) tous les fichiers du dossier `tokenizer_and_data/`
livré avec ce notebook :
- `tokenizer.json`
- `lm_train.bin`, `lm_valid.bin`, `lm_test.bin`
- `translation_tokenized_{train,valid,test}.jsonl`
- `dictionary_tokenized_{train,valid,test}.jsonl`


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/bariba_gpt_data')

Mounted at /content/drive


In [3]:
import os

file_path = "tokenizer.json"
if os.path.exists(file_path):
    print(f"Le fichier '{file_path}' est bien présent dans le répertoire courant : {os.getcwd()}")
else:
    print(f"Le fichier '{file_path}' n'a PAS été trouvé dans le répertoire courant : {os.getcwd()}")
    print("Veuillez vous assurer qu'il est bien présent dans '/content/drive/MyDrive/bariba_gpt_data' ou que le chemin d'accès est correct.")

Le fichier 'tokenizer.json' est bien présent dans le répertoire courant : /content/drive/MyDrive/bariba_gpt_data


In [4]:
!pip install -q tokenizers
import numpy as np, json, math, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cuda


## 1. Charger le tokenizer

C'est un tokenizer BPE byte-level entraîné spécifiquement sur ton corpus
bariba-français (vocab_size = 8000). Byte-level = aucun caractère bariba
(ɛ, ɔ, ɓ, ɗ, ŋ, ã, ĩ, tons...) ne peut être "inconnu" : tout se décompose
en octets si besoin, donc jamais de perte d'information.


In [5]:
tok = Tokenizer.from_file("tokenizer.json")
VOCAB_SIZE = tok.get_vocab_size()
PAD_ID = tok.token_to_id("<pad>")
BOS_ID = tok.token_to_id("<bos>")
EOS_ID = tok.token_to_id("<eos>")
BAR_ID = tok.token_to_id("<bariba>")
FR_ID  = tok.token_to_id("<fr>")
SEP_ID = tok.token_to_id("<sep>")
print("vocab_size:", VOCAB_SIZE)

# petit test
enc = tok.encode("Yè ba gberu da, bibu ba bɔ̃ɔ binun nɔni baakia.")
print(enc.ids)
print(tok.decode(enc.ids))


vocab_size: 8000
[659, 350, 1294, 429, 18, 1308, 350, 534, 269, 263, 3333, 288, 1172, 662, 763, 20]
Yè ba gberu da, bibu ba bɔ̃ɔ binun nɔni baakia.


## 2. Le modèle — GPT minimal

**C'est ici que se trouve l'embedding** : `nn.Embedding(vocab_size, d_model)`
transforme chaque identifiant de token en un vecteur de dimension `d_model`,
appris pendant l'entraînement (pas précalculé). On ajoute un embedding de
position, puis des blocs transformer standards (auto-attention causale +
MLP), puis une tête linéaire vers le vocabulaire.


In [6]:
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(),
            nn.Linear(4 * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=512, n_head=8, n_layer=6, dropout=0.0): # Modification: Dropout désactivé pour le test
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.position_embedding(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

## 3. Pré-entraînement type "langage" (next-token prediction)

Utilise `lm_train.bin` / `lm_valid.bin` : c'est tout le corpus (définitions
+ exemples, bariba et français mélangés) concaténé en une seule longue
séquence de tokens, au format nanoGPT (tableau uint16 sur disque).


In [7]:
BLOCK_SIZE = 256

train_data = np.memmap('lm_train.bin', dtype=np.uint16, mode='r')
valid_data = np.memmap('lm_valid.bin', dtype=np.uint16, mode='r')

def get_batch(data, batch_size=32, block_size=BLOCK_SIZE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

model = MiniGPT(VOCAB_SIZE, BLOCK_SIZE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(2000):
    xb, yb = get_batch(train_data)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 200 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch(valid_data)
            _, vloss = model(xv, yv)
        model.train()
        print(f"step {step}: train loss {loss.item():.3f}  valid loss {vloss.item():.3f}")

step 0: train loss 9.190  valid loss 8.088
step 200: train loss 4.138  valid loss 4.521
step 400: train loss 3.517  valid loss 4.058
step 600: train loss 2.732  valid loss 3.747
step 800: train loss 2.082  valid loss 3.743
step 1000: train loss 1.310  valid loss 4.007
step 1200: train loss 0.536  valid loss 4.318
step 1400: train loss 0.232  valid loss 4.542
step 1600: train loss 0.172  valid loss 4.599
step 1800: train loss 0.150  valid loss 4.950


## 4. Fine-tuning supervisé : traduction bariba ↔ français

Utilise `translation_tokenized_{train,valid,test}.jsonl` : chaque exemple a
`input_ids` (prompt + réponse) et `labels` (avec `-100` sur le prompt, donc
la perte ne compte que sur la partie à générer — traduction, ici).
Même mécanisme pour `dictionary_tokenized_*.jsonl` (définition d'un mot).


In [13]:
def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8')]

def collate(batch, pad_id=PAD_ID, max_len=BLOCK_SIZE):
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, ex in enumerate(batch):
        ids = ex['input_ids'][:max_len]
        lab = ex['labels'][:max_len]
        input_ids[i, :len(ids)] = torch.tensor(ids)
        labels[i, :len(lab)]    = torch.tensor(lab)
    return input_ids.to(device), labels.to(device)

train_pairs = load_jsonl('translation_tokenized_train.jsonl')
valid_pairs = load_jsonl('translation_tokenized_valid.jsonl')

import random
def get_finetune_batch(pairs, batch_size=32):
    batch = random.sample(pairs, batch_size)
    return collate(batch)

for step in range(3000): # Augmenter le nombre d'étapes de fine-tuning
    xb, yb = get_finetune_batch(train_pairs)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_finetune_batch(valid_pairs)
            _, vloss = model(xv, yv)
        model.train()
        print(f"step {step}: train loss {loss.item():.3f}  valid loss {vloss.item():.3f}")

step 0: train loss 13.059  valid loss 8.115
step 100: train loss 0.742  valid loss 1.069
step 200: train loss 0.261  valid loss 0.486
step 300: train loss 0.239  valid loss 0.269
step 400: train loss 0.150  valid loss 0.380
step 500: train loss 0.069  valid loss 0.218
step 600: train loss 0.036  valid loss 0.196
step 700: train loss 0.026  valid loss 0.118
step 800: train loss 0.012  valid loss 0.150
step 900: train loss 0.008  valid loss 0.156
step 1000: train loss 0.003  valid loss 0.146
step 1100: train loss 0.004  valid loss 0.088
step 1200: train loss 0.005  valid loss 0.143
step 1300: train loss 0.001  valid loss 0.082
step 1400: train loss 0.002  valid loss 0.099
step 1500: train loss 0.002  valid loss 0.076
step 1600: train loss 0.001  valid loss 0.045
step 1700: train loss 0.001  valid loss 0.048
step 1800: train loss 0.001  valid loss 0.079
step 1900: train loss 0.001  valid loss 0.152
step 2000: train loss 0.000  valid loss 0.034
step 2100: train loss 0.000  valid loss 0.086

## 5. Générer une traduction

Génération gloutonne simple (greedy) à partir d'un prompt formaté comme
pendant l'entraînement : `<bos> <bariba> ... <sep> <fr>`.


In [1]:
@torch.no_grad()
def translate(text, src_lang="bariba", max_new_tokens=10, beam_width=1): # Réduction de max_new_tokens et beam_width pour le débogage
    src_id, tgt_id = (BAR_ID, FR_ID) if src_lang == "bariba" else (FR_ID, BAR_ID)
    initial_ids = [BOS_ID, src_id] + tok.encode(text).ids + [SEP_ID, tgt_id]

    # Initialize beam: list of (score, sequence_ids)
    # scores are log probabilities, higher is better
    beam = [(0.0, torch.tensor([initial_ids], dtype=torch.long, device=device))]

    model.eval()
    print(f"Initial IDs: {initial_ids}")
    print(f"Decoded initial prompt: {tok.decode(initial_ids)}")

    generated_ids = []

    # Pour le débogage, nous connaissons la séquence attendue pour "Aa, a na kɔ." -> "Ah !"
    # Les IDs pour "Ah !" sont [39, 78, 1962]
    expected_output_ids = [39, 78, 1962, EOS_ID] # Ajout de EOS_ID à la fin de la séquence attendue

    for step_num in range(max_new_tokens):
        all_candidates = []
        for score, ids in beam:
            if ids[0, -1].item() == EOS_ID: # If sequence already ended, keep it
                all_candidates.append((score, ids))
                continue

            logits, _ = model(ids[:, -BLOCK_SIZE:])
            # Get log probabilities for the next token
            log_probs = F.log_softmax(logits[0, -1], dim=-1)

            # Get top 'beam_width' candidates for the next token
            # For debugging, let's get top 3 regardless of beam_width
            top_log_probs, top_indices = torch.topk(log_probs, k=3)

            print(f"\n--- Generation Step {step_num + 1} ---")
            print(f"Current sequence (ids): {ids.tolist()[0]}")
            print(f"Current sequence (decoded): {tok.decode(ids.tolist()[0])}")

            # Afficher la probabilité du token attendu
            if step_num < len(expected_output_ids):
                expected_token_id = expected_output_ids[step_num]
                expected_token_prob = torch.exp(log_probs[expected_token_id]).item()
                expected_token_str = tok.decode([expected_token_id])
                print(f"  -> Prob. du token attendu '{expected_token_str}' (ID: {expected_token_id}): {expected_token_prob:.4f}")

            print("Top 3 candidates for next token:")
            for i in range(3):
                token_id = top_indices[i].item()
                token_prob = torch.exp(top_log_probs[i]).item()
                token_str = tok.decode([token_id])
                print(f"  - ID: {token_id}, Token: '{token_str}', Prob: {token_prob:.4f}")

            # Apply beam_width logic only for actual beam selection
            current_beam_candidates = []
            for i in range(min(beam_width, len(top_indices))): # Use actual beam_width here
                next_token_log_prob = top_log_probs[i].item()
                next_token_id = top_indices[i].item()
                new_ids = torch.cat([ids, torch.tensor([[next_token_id]], device=device)], dim=1)
                new_score = score + next_token_log_prob
                current_beam_candidates.append((new_score, new_ids))
            all_candidates.extend(current_beam_candidates)

        # Select the top 'beam_width' candidates from all possibilities
        beam = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_width]

        # For greedy decoding (beam_width=1), the best_ids is simply the first one
        if beam_width == 1:
            generated_ids.append(beam[0][1][0, -1].item())

        # If all sequences in the beam have ended, stop early
        if all(ids[0, -1].item() == EOS_ID for score, ids in beam):
            break

    # Pick the best sequence from the beam (highest score)!
    best_score, best_ids = beam[0]

    print(f"\nFinal best sequence IDs: {best_ids.tolist()[0]}")
    print(f"Final decoded best sequence: {tok.decode(best_ids[0].tolist())}")
    return tok.decode(best_ids[0].tolist())

# Nous allons tester une phrase Bariba prise directement de votre jeu de données d'entraînement (Exemple 1)
# La traduction attendue est 'Ah !'
print(translate("Aa, a na kɔ.", src_lang="bariba", beam_width=1)) # Forcing greedy decoding for debugging

NameError: name 'torch' is not defined

In [26]:
# Testons la perte et les probabilités du modèle pour l'exemple d'entraînement problématique
# Cet exemple est train_pairs[0]

# Assurez-vous que train_pairs est chargé
if 'train_pairs' not in locals():
    train_pairs = load_jsonl('translation_tokenized_train.jsonl')

example_data = train_pairs[0]

# Convertir l'exemple en un mini-batch (taille 1)
# Utilisons le même mécanisme de collation que pendant l'entraînement
example_batch_input, example_batch_labels = collate([example_data], max_len=BLOCK_SIZE)

print("--- Test de perte pour un exemple spécifique d'entraînement ---")
print(f"Input IDs pour le batch de test: {example_batch_input.tolist()}")
print(f"Labels pour le batch de test: {example_batch_labels.tolist()}")

model.train() # Mettre le modèle en mode entraînement pour une évaluation correcte de la perte
with torch.no_grad(): # Pas besoin de gradients ici, juste pour le calcul de la perte
    logits, loss = model(example_batch_input, example_batch_labels)

print(f"\nPerte calculée pour cet exemple: {loss.item():.6f}")

# Maintenant, inspectons les logits et probabilités pour le premier token cible
# La séquence de prompt qui mène au premier token cible est:
# [BOS_ID, BAR_ID, <tokens for 'Aa, a na kɔ.'>, SEP_ID, FR_ID]
# C'est la portion de example_batch_input avant les labels non masqués.

# Trouver l'indice du premier token non masqué dans les labels
first_label_idx = (example_batch_labels[0] != -100).nonzero(as_tuple=True)[0][0].item()

# Les logits pertinents sont ceux à cet index
logits_for_first_target_token = logits[0, first_label_idx]

# Calculer les probabilités à partir de ces logits
probabilities = F.softmax(logits_for_first_target_token, dim=-1)

# Le token cible attendu est example_batch_labels[0, first_label_idx]
expected_target_token_id = example_batch_labels[0, first_label_idx].item()
expected_target_token_str = tok.decode([expected_target_token_id])
expected_target_token_prob = probabilities[expected_target_token_id].item()

print(f"\nProbabilité du token attendu '{expected_target_token_str}' (ID: {expected_target_token_id}) après entraînement: {expected_target_token_prob:.4f}")

# Afficher les 3 meilleurs candidats du modèle pour cette position
top_probs, top_indices = torch.topk(probabilities, k=3)
print("Top 3 candidats prédits pour cette position:")
for i in range(3):
    token_id = top_indices[i].item()
    token_prob = top_probs[i].item()
    token_str = tok.decode([token_id])
    print(f"  - ID: {token_id}, Token: '{token_str}', Prob: {token_prob:.4f}")

print("------------------------------------------------------------------")

--- Test de perte pour un exemple spécifique d'entraînement ---
Input IDs pour le batch de test: [[2, 4, 39, 71, 18, 286, 456, 499, 20, 6, 5, 39, 78, 1962, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
Labels pour le batch de test: [[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 39, 78, 1962, 3, -1

In [24]:
# Vérifions la tokenisation de la traduction attendue 'Ah !'
expected_translation = "Ah !"
encoded_expected = tok.encode(expected_translation)

print(f"Traduction attendue: '{expected_translation}'")
print(f"IDs de la traduction attendue: {encoded_expected.ids}")
print(f"Tokens de la traduction attendue: {tok.decode(encoded_expected.ids)}")

Traduction attendue: 'Ah !'
IDs de la traduction attendue: [39, 78, 1962]
Tokens de la traduction attendue: Ah !


In [21]:
# Affichons plusieurs exemples de paires input_ids et labels du jeu de données d'entraînement
# pour vérifier la structure et le contenu.

# Assurez-vous que train_pairs est chargé (cela devrait être le cas après la cellule add639fc)
if 'train_pairs' not in locals():
    train_pairs = load_jsonl('translation_tokenized_train.jsonl')

print("--- Exemples de données d'entraînement (paires input_ids et labels) ---")

# Afficher les 10 premiers exemples pour une inspection plus approfondie
for i in range(10):
    if i >= len(train_pairs):
        break
    example = train_pairs[i]

    # Décoder les IDs pour une meilleure lisibilité
    # Notez que les labels contiennent -100 pour les tokens non pertinents pour le calcul de la perte.
    # Nous allons filtrer les -100 pour décoder.

    # Le 'input' décodé contiendra la concaténation de la source et de la cible comme dans l'entraînement.
    decoded_full_input_sequence = tok.decode([id for id in example['input_ids'] if id != -100])
    # Le 'label' décodé contiendra la traduction cible attendue par le modèle.
    decoded_label_target = tok.decode([id for id in example['labels'] if id != -100])

    print(f"\nExemple {i+1}:")
    print(f"Raw input_ids: {example['input_ids'][:20]}...") # Truncate for readability
    print(f"Raw labels: {example['labels'][:20]}...")      # Truncate for readability
    print(f"Decoded full sequence (input for model): {decoded_full_input_sequence}")
    print(f"Decoded label (traduction attendue): {decoded_label_target}")

    try:
        # Identifier les IDs de la langue source et cible à partir de l'exemple
        source_lang_id_in_example = example['input_ids'][1]
        sep_idx = example['input_ids'].index(SEP_ID)
        target_lang_id_in_example = example['input_ids'][sep_idx + 1]

        # Extraire le texte source et cible en utilisant les indices et les IDs de langue
        src_text_ids = [id for id in example['input_ids'][2:sep_idx] if id != -100]
        tgt_text_ids = [id for id in example['input_ids'][sep_idx + 2 : -1] if id != -100]

        decoded_src_text = tok.decode(src_text_ids)
        decoded_tgt_text = tok.decode(tgt_text_ids)

        src_lang_name = "Bariba" if source_lang_id_in_example == BAR_ID else "Français"
        tgt_lang_name = "Bariba" if target_lang_id_in_example == BAR_ID else "Français"

        print(f"  -> Source ({src_lang_name}): {decoded_src_text}")
        print(f"  -> Cible ({tgt_lang_name}): {decoded_tgt_text}")
        print(f"  -> (Note: Le 'Decoded label' doit correspondre à la 'Cible' ici.)")

        # *** Nouvelle vérification: Source et Cible sont-elles identiques ? ***
        if decoded_src_text.strip().lower() == decoded_tgt_text.strip().lower():
            print("  !!! ATTENTION : Le texte source et le texte cible sont identiques !!!")

    except ValueError:
        print("  (Impossible de séparer clairement les parties source et cible dans input_ids)")

print("------------------------------------------------------------------")

--- Exemples de données d'entraînement (paires input_ids et labels) ---

Exemple 1:
Raw input_ids: [2, 4, 39, 71, 18, 286, 456, 499, 20, 6, 5, 39, 78, 1962, 3]...
Raw labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 39, 78, 1962, 3]...
Decoded full sequence (input for model): Aa, a na kɔ.Ah !
Decoded label (traduction attendue): Ah !
  -> Source (Bariba): Aa, a na kɔ.
  -> Cible (Français): Ah !
  -> (Note: Le 'Decoded label' doit correspondre à la 'Cible' ici.)

Exemple 2:
Raw input_ids: [2, 5, 39, 78, 1962, 6, 4, 39, 71, 18, 286, 456, 499, 20, 3]...
Raw labels: [-100, -100, -100, -100, -100, -100, -100, 39, 71, 18, 286, 456, 499, 20, 3]...
Decoded full sequence (input for model): Ah !Aa, a na kɔ.
Decoded label (traduction attendue): Aa, a na kɔ.
  -> Source (Français): Ah !
  -> Cible (Bariba): Aa, a na kɔ.
  -> (Note: Le 'Decoded label' doit correspondre à la 'Cible' ici.)

Exemple 3:
Raw input_ids: [2, 4, 39, 269, 71, 1425, 18, 286, 985, 495, 469, 805, 6, 

## Où sont les "vecteurs" (embeddings) ?

Après entraînement, la matrice apprise `model.token_embedding.weight` est
de taille `(vocab_size, d_model)` : une ligne = le vecteur appris pour un
token. On peut l'extraire à tout moment :

```python
embedding_matrix = model.token_embedding.weight.detach().cpu().numpy()
# embedding_matrix.shape == (8000, 256)
```

C'est ça, "l'embedding du dataset" — mais il n'existe qu'**après** avoir
entraîné le modèle, pas avant. Avant l'entraînement, tout ce qu'on peut
préparer, c'est le tokenizer et les identifiants (ce que ce notebook fait).


In [12]:
import torch

# Définissez le chemin où vous voulez sauvegarder votre modèle
model_save_path = '/content/drive/MyDrive/bariba_gpt_data/minigpt_bariba_fr_v2.pth'

# Sauvegarder les poids du modèle
torch.save(model.state_dict(), model_save_path)

print(f"Le modèle a été sauvegardé avec succès à : {model_save_path}")

Le modèle a été sauvegardé avec succès à : /content/drive/MyDrive/bariba_gpt_data/minigpt_bariba_fr_v2.pth
